# Multimodal Meme Sexism Detection

Arquitectura: **XLM-RoBERTa** + **GatedTextFusion** + dos ramas **CrossModalAttention** (EEG | ET+HR).  
Entrenamiento en dos fases con **SoftLabelLoss** (KLDiv) sobre distribuciones de anotadores.

| Celda | Contenido |
|---|---|
| 1 | Configuración global e imports |
| 2 | `TextEncoder` |
| 3 | `GatedTextFusion` + `CrossModalAttentionBranch` |
| 4 | `MultimodalModel` |
| 5 | `SoftLabelLoss` |
| 6 | `MemeDataset` + utilidades |
| 7 | `collate_fn` |
| 8 | Carga de datos y DataLoaders |
| 9 | `evaluate` + `EarlyStopping` |
| 10 | Loop de entrenamiento (`train`) |
| 11 | Lanzar entrenamiento |

## 1 · Configuración global e imports

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import roc_auc_score, f1_score

# ── Constantes ────────────────────────────────────────────────────────────────
TEXT_ENCODER_NAME = "xlm-roberta-base"
MAX_SUBJECTS      = 4
TEXT_DIM          = 768
NUM_CLASSES       = 2        # 0 = no-sexist | 1 = sexist

OCR_LEN = 128
TRANS_LEN = 128
REAS_LEN = 256

SEQ_LEN = [OCR_LEN, TRANS_LEN, REAS_LEN]

PREDS_DIR  = "../data/processed/predictions/"
SAVE_DIR   = "../data/processed/"
DATA_DIR   = "../data/processed/"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2 · TextEncoder

In [2]:
class TextEncoder(nn.Module):
    """
    Wrapper de XLM-RoBERTa.
    - freeze=True  : backbone congelado (Fase 1)
    - freeze=False : fine-tune completo (Fase 2)
    """

    def __init__(self, model_name: str = TEXT_ENCODER_NAME, freeze: bool = True):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.freeze_backbone(freeze)

    def freeze_backbone(self, freeze: bool):
        for p in self.model.parameters():
            p.requires_grad = not freeze

    def get_sequential_output(self, input_ids, attention_mask):
        """Devuelve [B, seq_len, 768]."""
        return self.model(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

## 3 · GatedTextFusion + CrossModalAttentionBranch

In [3]:
class GatedTextFusion(nn.Module):
    def __init__(self, text_dim: int = TEXT_DIM, seg_lengths: list = [128, 128, 256], dropout: float = 0.1):
        super().__init__()
        self.seg_lengths = seg_lengths
        self.n_segments  = len(seg_lengths)
        self.gate_net = nn.Sequential(
            nn.Linear(text_dim, text_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(text_dim // 2, self.n_segments),
        )

    def forward(self, text_seq: torch.Tensor) -> torch.Tensor:
        """text_seq: [B, 512, D]  →  [B, D]"""
        gates    = torch.softmax(self.gate_net(text_seq[:, 0, :]), dim=-1)  # [B, n_seg]

        segments = []
        start = 0
        for length in self.seg_lengths:
            segments.append(text_seq[:, start:start + length, :].mean(dim=1))  # [B, D]
            start += length

        seg_stack = torch.stack(segments, dim=1)                # [B, n_seg, D]
        return (seg_stack * gates.unsqueeze(-1)).sum(dim=1)     # [B, D]


class CrossModalAttentionBranch(nn.Module):
    """
    Rama fisiológica independiente (EEG o ET+HR).

    physio [B, MAX_S, physio_dim]
        ↓  MLP Projection  →  [B, MAX_S, D]
        ↓  Cross-Attention  Q=physio, K=V=text_seq
        ↓  MLP Importancia + Softmax  (mask padding → -inf)
    [B, D]
    """

    def __init__(self, physio_dim: int, text_dim: int = TEXT_DIM,
                 num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.physio_proj = nn.Sequential(
            nn.Linear(physio_dim, text_dim),
            nn.LayerNorm(text_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=text_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True,
        )
        self.attn_norm      = nn.LayerNorm(text_dim)
        self.importance_mlp = nn.Sequential(
            nn.Linear(text_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, physio_seq: torch.Tensor, text_seq: torch.Tensor,
                physio_mask: torch.Tensor) -> torch.Tensor:
        """
        physio_seq  : [B, MAX_S, physio_dim]
        text_seq    : [B, seq_len, D]
        physio_mask : [B, MAX_S]  True=real, False=padding
        Returns     : [B, D]
        """
        p           = self.physio_proj(physio_seq)
        attn_out, _ = self.cross_attn(query=p, key=text_seq, value=text_seq)
        p_attended  = self.attn_norm(p + attn_out) * physio_mask.unsqueeze(-1).float()

        scores  = self.importance_mlp(p_attended)
        scores  = scores.masked_fill(~physio_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return (p_attended * weights).sum(dim=1)

## 4 · MultimodalModel

In [4]:
class MultimodalModel(nn.Module):
    """
    Arquitectura completa:

        Text (OCR </s> TRANSCRIPTION </s> REASONING)
            ↓  XLM-RoBERTa  →  text_seq [B, seq_len, 768]
            ↓  GatedTextFusion           →  text_fused  [B, 768]

        EEG   → CrossModalAttentionBranch(text_seq) →  eeg_ctx   [B, 768]
        ET+HR → CrossModalAttentionBranch(text_seq) →  ethr_ctx  [B, 768]

        [cls_token ‖ text_fused ‖ eeg_ctx ‖ ethr_ctx]  →  [B, 3072]
            ↓  MLP clasificador
        logits [B, 2]
    """

    def __init__(
        self,
        eeg_dim:         int,
        et_hr_dim:       int,
        text_dim:        int   = TEXT_DIM,
        num_heads:       int   = 8,
        seg_lengths: list=[128,128,256],
        freeze_backbone: bool  = True,
        dropout:         float = 0.1,
    ):
        super().__init__()
        self.text_encoder = TextEncoder(freeze=freeze_backbone)
        self.text_gate = GatedTextFusion(text_dim=768, seg_lengths=seg_lengths, dropout=dropout)
        self.eeg_branch   = CrossModalAttentionBranch(eeg_dim,   text_dim, num_heads, dropout)
        self.et_hr_branch = CrossModalAttentionBranch(et_hr_dim, text_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(text_dim * 4, 256),   # 3072 → 256  (antes 512)
            nn.LayerNorm(256),
            nn.GELU(),                       # GELU en lugar de ReLU: gradientes más suaves
            nn.Dropout(0.5),                 # antes 0.4
            nn.Linear(256, 64),              # antes 128
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(
        self,
        input_ids:      torch.Tensor,   # [B, seq_len]
        attention_mask: torch.Tensor,   # [B, seq_len]
        eeg:            torch.Tensor,   # [B, MAX_S, eeg_dim]
        eeg_mask:       torch.Tensor,   # [B, MAX_S]
        et_hr:          torch.Tensor,   # [B, MAX_S, et_hr_dim]
        et_hr_mask:     torch.Tensor,   # [B, MAX_S]
    ) -> torch.Tensor:                  # [B, 2]
        text_seq   = self.text_encoder.get_sequential_output(input_ids, attention_mask)
        cls_token  = text_seq[:, 0, :]
        text_fused = self.text_gate(text_seq)
        eeg_ctx    = self.eeg_branch(eeg,   text_seq, eeg_mask)
        et_hr_ctx  = self.et_hr_branch(et_hr, text_seq, et_hr_mask)
        fused      = torch.cat([cls_token, text_fused, eeg_ctx, et_hr_ctx], dim=1)
        return self.classifier(fused)

## 5 · SoftLabelLoss

KL-Divergence sobre distribuciones de anotadores en lugar de CrossEntropy con hard labels.  
Un meme anotado [1,1,1,0] genera soft_label=[0.25, 0.75]; el modelo es penalizado
*proporcionalmente* a su desacuerdo con esa distribución, no de forma binaria.

In [5]:
class SoftLabelLoss(nn.Module):
    """KLDiv con label smoothing opcional sobre la distribución target."""
 
    def __init__(self, smoothing: float = 0.1, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.kl          = nn.KLDivLoss(reduction="batchmean")
 
    def forward(self, logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
        # Mezcla la distribución de anotadores con la uniforme
        # soft_targets_smooth = (1 - ε) * soft_targets + ε * (1/K)
        uniform = torch.full_like(soft_targets, 1.0 / self.num_classes)
        smoothed = (1 - self.smoothing) * soft_targets + self.smoothing * uniform
        return self.kl(F.log_softmax(logits, dim=-1), smoothed)

## 6 · MemeDataset + utilidades

**Decisiones de diseño:**
- `qwen_label` **excluido** del texto — es leakage directo en training.
- Labels como **soft_label** `[p_no_sexist, p_sexist]` derivados de `votes`.  
  Si no hay `votes`, se usa one-hot del campo `label` como fallback.

In [6]:
def pad_subjects(subject_list: list, max_subjects: int, feat_dim: int):
    """Lista de vectores → tensor [max_subjects, feat_dim] + máscara booleana."""
    tensor = torch.zeros(max_subjects, feat_dim)
    mask   = torch.zeros(max_subjects, dtype=torch.bool)
    for i, s in enumerate(subject_list[:max_subjects]):
        tensor[i] = torch.tensor(s, dtype=torch.float)
        mask[i]   = True
    return tensor, mask


def votes_to_soft_label(votes: list, num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """[1,1,1,0]  →  tensor([0.25, 0.75])"""
    counts = torch.zeros(num_classes)
    for v in votes:
        counts[v] += 1
    return counts / counts.sum()


class MemeDataset(Dataset):
    """
    Campos esperados por sample:
        qwen_ocr           : str | None
        qwen_transcription : str | None
        qwen_reasoning     : str | None
        physio             : {"EEG": [...], "ET": [...], "HR": [...]}
        votes              : list[int]   (preferido: [0,1,1,0])
        label              : int         (fallback si no hay votes)
    """

    def __init__(self, data, tokenizer, eeg_dim=None, et_hr_dim=None,
             ocr_len=128, trans_len=128, reasoning_len=256):          # ← nuevos parámetros
        self.data          = data
        self.tokenizer     = tokenizer
        self.ocr_len       = ocr_len
        self.trans_len     = trans_len
        self.reasoning_len = reasoning_len

        first = data[0]["physio"]
        eeg_s = first.get("EEG", [])
        et_s  = first.get("ET",  [])
        hr_s  = first.get("HR",  [])

        self.eeg_dim   = eeg_dim   or (len(eeg_s[0]) if eeg_s else 1)
        et_dim         = len(et_s[0]) if et_s else 0
        hr_dim         = len(hr_s[0]) if hr_s else 0
        self.et_hr_dim = et_hr_dim or (et_dim + hr_dim) or 1

        print(f"[MemeDataset] EEG_DIM={self.eeg_dim} | ET_HR_DIM={self.et_hr_dim} | n={len(data)}")

    def _encode_part(self, text: str | None, prefix: str, max_length: int) -> tuple:
        """Tokeniza una sola parte; devuelve (input_ids, attention_mask) como tensores 1-D."""
        content = f"{prefix} {text}" if text else ""
        enc = self.tokenizer(
            content,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        ocr_ids,   ocr_mask   = self._encode_part(sample.get("qwen_ocr"),           "[OCR]",           self.ocr_len)
        trans_ids, trans_mask = self._encode_part(sample.get("qwen_transcription"),  "[TRANSCRIPTION]", self.trans_len)
        reas_ids,  reas_mask  = self._encode_part(sample.get("qwen_reasoning"),      "[REASONING]",     self.reasoning_len)

        # ── 2. Concatenación → [ocr_len + trans_len + reasoning_len] ──────────
        input_ids      = torch.cat([ocr_ids,   trans_ids,  reas_ids],  dim=0)   # [768]
        attention_mask = torch.cat([ocr_mask,  trans_mask, reas_mask], dim=0)   # [768]

        # ── 2. Fisiología ─────────────────────────────────────────────────
        physio  = sample.get("physio", {})
        eeg_s   = physio.get("EEG", [])
        et_s    = physio.get("ET",  [])
        hr_s    = physio.get("HR",  [])

        n       = max(len(et_s), len(hr_s))
        et_dim  = len(et_s[0]) if et_s else 0
        hr_dim  = len(hr_s[0]) if hr_s else 0
        et_hr   = [
            (et_s[i] if i < len(et_s) else [0.0]*et_dim) +
            (hr_s[i] if i < len(hr_s) else [0.0]*hr_dim)
            for i in range(n)
        ]

        eeg_seq,   eeg_mask   = pad_subjects(eeg_s, MAX_SUBJECTS, self.eeg_dim)
        et_hr_seq, et_hr_mask = pad_subjects(et_hr, MAX_SUBJECTS, self.et_hr_dim)

        # ── 3. Soft label ─────────────────────────────────────────────────
        soft_label = votes_to_soft_label(sample["label"])

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "eeg":            eeg_seq,
            "eeg_mask":       eeg_mask,
            "et_hr":          et_hr_seq,
            "et_hr_mask":     et_hr_mask,
            "soft_label":     soft_label,   # [2]
        }

## 7 · collate_fn

In [7]:
def collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "eeg":            torch.stack([b["eeg"]            for b in batch]),
        "eeg_mask":       torch.stack([b["eeg_mask"]       for b in batch]),
        "et_hr":          torch.stack([b["et_hr"]          for b in batch]),
        "et_hr_mask":     torch.stack([b["et_hr_mask"]     for b in batch]),
        "soft_label":     torch.stack([b["soft_label"]     for b in batch]),   # [B, 2]
    }

## 8 · Carga de datos y DataLoaders

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME)

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_data = json.load(f)

with open(os.path.join(DATA_DIR, "val.json"), encoding="utf-8") as f:
    val_data = json.load(f)



train_dataset = MemeDataset(train_data, tokenizer, 
                            ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN)
val_dataset   = MemeDataset(val_data,   tokenizer,
                            eeg_dim=train_dataset.eeg_dim,
                            et_hr_dim=train_dataset.et_hr_dim,
                            ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN)

eeg_dim   = train_dataset.eeg_dim
et_hr_dim = train_dataset.et_hr_dim

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=1705
[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=301
Train batches: 214 | Val batches: 19


## 9 · evaluate + EarlyStopping

In [9]:
def evaluate(model, criterion, loader, device, save_path=None):
    """
    Evalúa sobre `loader` y devuelve (val_loss, auc, f1_macro, f1_yes).

    - Pérdida calculada sobre soft_label (consistente con training).
    - Hard label para métricas: argmax(soft_label).
    - Guarda predicciones en JSON solo si f1_macro mejora el registro previo.
    """
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            soft_labels = batch["soft_label"].to(device)
            total_loss += criterion(logits, soft_labels).item()

            all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_labels.extend(batch["soft_label"].argmax(dim=1).numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= 0.5).astype(int)

    auc      = roc_auc_score(all_labels, all_probs)
    f1_macro = f1_score(all_labels, preds, average="macro")
    f1_yes   = f1_score(all_labels, preds, pos_label=1, average="binary")

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        new_metrics = {
            "auc":      round(float(auc),      4),
            "f1_macro": round(float(f1_macro),  4),
            "f1_yes":   round(float(f1_yes),    4),
        }
        save = True
        if os.path.exists(save_path):
            with open(save_path) as f:
                save = new_metrics["f1_macro"] > json.load(f).get("metrics", {}).get("f1_macro", -1)
        if save:
            print("  [evaluate] Guardando predicciones...")
            with open(save_path, "w") as f:
                json.dump({
                    "predictions": [
                        {"label": int(l), "pred": int(p), "prob": float(prob)}
                        for l, p, prob in zip(all_labels, preds, all_probs)
                    ],
                    "metrics": new_metrics,
                }, f, indent=2)

    return total_loss / len(loader), auc, f1_macro, f1_yes


class EarlyStopping:
    """
    Detiene el entrenamiento si F1-macro no mejora en `patience` épocas.
    Guarda automáticamente el mejor checkpoint.
    """

    def __init__(self, patience: int = 4, min_delta: float = 1e-4, save_path: str = "best_model.pt"):
        self.patience  = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.best_f1   = -1.0
        self.counter   = 0

    def step(self, f1: float, model: nn.Module) -> bool:
        """Devuelve True si hay que parar."""
        if f1 > self.best_f1 + self.min_delta:
            self.best_f1 = f1
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

## 10 · Función `train`

**Fase 1** — backbone congelado; solo se entrenan gate, ramas fisiológicas y clasificador.  
**Fase 2** — fine-tune completo con LR discriminativos por capa del encoder.

In [10]:
def train(
    train_data,
    loader,
    val_loader,
    eeg_dim:           int,
    et_hr_dim:         int,
    text_encoder_name: str   = TEXT_ENCODER_NAME,
    save_dir:          str   = SAVE_DIR,
    phase1_epochs:     int   = 5,
    phase2_epochs:     int   = 10,
    es_patience:       int   = 4,
):
    os.makedirs(save_dir, exist_ok=True)
    json_path  = os.path.join(PREDS_DIR, f"{text_encoder_name}.json")
    model_path = os.path.join(save_dir,  f"{text_encoder_name}.pt")

    model = MultimodalModel(
        eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
        text_dim=768, num_heads=8, freeze_backbone=True,
        seg_lengths=SEQ_LEN
    ).to(device)

    criterion = SoftLabelLoss()

    # ── Fase 1: backbone congelado ────────────────────────────────────────
    print("\n=== FASE 1: backbone congelado ===\n")

    params1    = [p for n, p in model.named_parameters() if "text_encoder" not in n and p.requires_grad]
    optimizer1 = torch.optim.AdamW(params1, lr=1e-5, weight_decay=0.05)
    scheduler1 = CosineAnnealingLR(optimizer1, T_max=phase1_epochs, eta_min=1e-6)

    for epoch in range(phase1_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph1 {epoch+1}/{phase1_epochs}")
        for batch in pbar:
            optimizer1.zero_grad()
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            loss = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer1.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler1.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}")

    # ── Fase 2: fine-tune con LR discriminativos ──────────────────────────
    print("\n=== FASE 2: fine-tune con LR discriminativos ===\n")

    model.text_encoder.freeze_backbone(False)
    enc_layers = list(model.text_encoder.model.encoder.layer)
    n          = len(enc_layers)

    param_groups = [
        {"params": model.text_encoder.model.embeddings.parameters(), "lr": 1e-6},   # antes 2e-6
        *[{"params": l.parameters(), "lr": 1e-6} for l in enc_layers[:n//2]],       # antes 2e-6
        *[{"params": l.parameters(), "lr": 5e-6} for l in enc_layers[n//2:]],       # antes 1e-5
        {"params": [p for name, p in model.named_parameters()
                    if "text_encoder" not in name], "lr": 2e-5},                    # antes 5e-5
    ]
    optimizer2 = torch.optim.AdamW(param_groups, weight_decay=0.05)
    scheduler2 = CosineAnnealingLR(optimizer2, T_max=phase2_epochs, eta_min=1e-8)
    early_stop = EarlyStopping(patience=es_patience, save_path=model_path)

    best_f1 = 0.0
    for epoch in range(phase2_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph2 {epoch+1}/{phase2_epochs}")
        for batch in pbar:
            optimizer2.zero_grad()
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            loss = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer2.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler2.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        marker = " ← best" if f1 > best_f1 else ""
        best_f1 = max(best_f1, f1)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}{marker}")

        if early_stop.step(f1, model):
            print(f"  [EarlyStopping] Sin mejora en {es_patience} épocas. Parando.")
            break

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"\n✅ Mejor modelo cargado  (F1={early_stop.best_f1:.4f})")

    return model

## 11 · Lanzar entrenamiento

In [11]:
model = train(
    train_data        = train_data,
    loader            = train_loader,
    val_loader        = val_loader,
    eeg_dim           = eeg_dim,
    et_hr_dim         = et_hr_dim,
    text_encoder_name = TEXT_ENCODER_NAME,
    phase1_epochs=5,
    phase2_epochs=10,
    es_patience=5
)


=== FASE 1: backbone congelado ===



Ph1 1/5: 100%|██████████| 214/214 [00:17<00:00, 12.50it/s, loss=0.2783]


  → AUC=0.6040  F1=0.3874  F1_yes=0.0903  loss=0.3937


Ph1 2/5: 100%|██████████| 214/214 [00:17<00:00, 12.57it/s, loss=0.5047]


  → AUC=0.6307  F1=0.4861  F1_yes=0.2727  loss=0.3891


...

  → AUC=0.8014  F1=0.7202  F1_yes=0.7063  loss=0.3152


Ph2 9/10: 100%|██████████| 214/214 [00:48<00:00,  4.40it/s, loss=0.0131]


  → AUC=0.8012  F1=0.7198  F1_yes=0.7021  loss=0.3168


Ph2 10/10: 100%|██████████| 214/214 [00:48<00:00,  4.38it/s, loss=1.0552]


  → AUC=0.8019  F1=0.7200  F1_yes=0.7042  loss=0.3150

✅ Mejor modelo cargado  (F1=0.7308)


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# 12 · Evaluación sobre test
# ══════════════════════════════════════════════════════════════════════════════
# Requisitos previos (ya definidos en celdas anteriores):
#   - model       : entrenado y con el mejor checkpoint cargado
#   - criterion   : SoftLabelLoss()
#   - tokenizer, collate_fn, MemeDataset
#   - eeg_dim, et_hr_dim : inferidos del train_dataset

with open(os.path.join(DATA_DIR, "test.json"), encoding="utf-8") as f:
    test_data = json.load(f)

test_dataset = MemeDataset(
    test_data, tokenizer,
    eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)

# ── Evaluación ────────────────────────────────────────────────────────────────
test_save_path = os.path.join(PREDS_DIR, f"{TEXT_ENCODER_NAME}_test.json")

test_loss, test_auc, test_f1, test_f1_yes = evaluate(
    model, SoftLabelLoss(), test_loader, device, save_path=test_save_path
)

print("\n" + "═" * 50)
print("  TEST RESULTS")
print("═" * 50)
print(f"  Loss   : {test_loss:.4f}")
print(f"  AUC    : {test_auc:.4f}")
print(f"  F1     : {test_f1:.4f}")
print(f"  F1_yes : {test_f1_yes:.4f}")
print("═" * 50)
print(f"  Predicciones guardadas → {test_save_path}")

[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=502



══════════════════════════════════════════════════
  TEST RESULTS
══════════════════════════════════════════════════
  Loss   : 0.2811
  AUC    : 0.8020
  F1     : 0.7389
  F1_yes : 0.7456
══════════════════════════════════════════════════
  Predicciones guardadas → ../data/processed/predictions/xlm-roberta-base_test.json
